In [0]:
from pyspark.sql import functions as F

studies_df = spark.read.table(
    "clinical_trials.silver_studies"
)

conditions_df = spark.read.table(
    "clinical_trials.silver_conditions"
)

interventions_df = spark.read.table(
    "clinical_trials.silver_interventions"
)

outcomes_df = spark.read.table(
    "clinical_trials.silver_outcomes"
)

locations_df = spark.read.table(
    "clinical_trials.silver_locations"
)

In [0]:
# Silver에서는 분석 및 정규화를 위해 Study-Condition,
# Study-Location 관계를 별도 row로 관리한다.
#
# Serving Layer에서는 application의 조회 편의를 위해
# 관련 condition/country를 study 단위의 array로 다시 집계한다.

condition_summary_df = (
    conditions_df
    .groupBy("nct_id")
    .agg(
        F.sort_array(
            # collect_set : collect_list()와 달리 중복을 제거하면서 배열로 모읍니다.
            F.collect_set("condition")
        ).alias("conditions")
    )
)

country_summary_df = (
    locations_df
    .filter(F.col("country").isNotNull())
    .groupBy("nct_id")
    .agg(
        F.sort_array(
            F.collect_set("country")
        ).alias("countries")
    )
)

In [0]:
serving_study_search_df = (
    studies_df
    .join(
        condition_summary_df,
        on="nct_id",
        how="left"
    )
    .join(
        country_summary_df,
        on="nct_id",
        how="left"
    )
    .select(
        "nct_id",
        "brief_title",
        "official_title",
        "study_type",
        "phases",
        "overall_status",
        "enrollment_count",
        "start_date",
        "completion_date",
        "last_update_date",
        "has_results",
        "conditions",
        "countries"
    )
)

display(
    serving_study_search_df.limit(20)
)

nct_id,brief_title,official_title,study_type,phases,overall_status,enrollment_count,start_date,completion_date,last_update_date,has_results,conditions,countries
NCT00634335,Echocardiographic Characteristics of High Endurance Israeli Athletes,Echocardiographic Characteristics of High Endurance Israeli Athletes,OBSERVATIONAL,null,UNKNOWN,81,2008-03-01,2009-06-01,2008-12-30,false,List(Athlete's Heart),List(Israel)
NCT05604235,Effectiveness of Oncological Physiotherapy on Painful Shoulder Cervical Ganglion in Head and Neck Cancer,Effectiveness of Oncological Physiotherapy With Neurodynamics and Kinesitherapy (Active+Passive) on Painful Shoulder Cervical Ganglion Post Cervical Evacuation in Head and Neck Cancer Due to Accessory Spinal Nerve of the Accessory Spinal Nerve,INTERVENTIONAL,List(NA),COMPLETED,20,2022-11-15,2024-11-15,2024-11-20,false,"List(Cancer, Rehabilitation, Shoulder Pain)",List(Spain)
NCT06927635,Stereotactic Body Radiotherapy or Standard of Care for Prostate Oligoprogressive Cancer,Stereotactic Body Radiotherapy or Standard of Care for Prostate Oligoprogressive Cancer: a Randomized Phase II Trial,INTERVENTIONAL,List(PHASE2),RECRUITING,75,2025-07-08,2030-06-01,2026-07-07,false,"List(Castrate Resistance Prostate Cancer, OligoProgressive Metastatic Disease)",List(Canada)
NCT06362135,Virtual Home Visits for Stroke Patients and Their Caregivers,The Evaluation of the Effectiveness of Virtual Home Visits for Stroke Patients and Their Caregivers as Part of Home Health Services.,INTERVENTIONAL,List(NA),UNKNOWN,96,2023-08-01,2025-03-01,2024-04-15,false,"List(Caregiver, Home Visit, Stroke, Virtual)",List(Turkey (Türkiye))
NCT01805635,PF and FeNO and Clinical Characteristics in Children Being Diagnosed Due to Suspicion of Allergic Diseases,Evaluation of the Relationship Between the Parameters of Pulmonary Function (PF) and the Concentration of Fractional Exhaled Nitric Oxide (FeNO) and Clinical Characteristics in Children Being Diagnosed Due to Suspicion of Allergic Diseases,OBSERVATIONAL,null,UNKNOWN,5500,2013-02-01,2014-04-01,2013-09-09,false,"List(Allergic Rhinitis, Asthma, Atopic Dermatitis, Healthy, Urticaria)",List(Poland)
NCT01228435,IPI-504 in NSCLC Patients With ALK Translocations,"A Phase II Study of IPI-204, A Novel Hsp90 Inhibitor in NSCLC Patients With ALK Translocations",INTERVENTIONAL,List(PHASE2),TERMINATED,3,2010-10-01,2012-02-01,2017-12-04,true,"List(Lung Cancer, Stage IIIb Lung Cancer, Stage IV Lung Cancer)",List(United States)
NCT01748435,Pre-emptive Analgesia With Qutenza in Lower Limb Amputation,The Role of Pre-emptive Analgesia With Qutenza (Topical Capsaicin 8%) in Preventing Neuropathic Pain Following Lower Limb Amputation: a Pilot Randomised Controlled Study,OBSERVATIONAL,null,UNKNOWN,30,2013-02-01,2014-05-01,2012-12-12,false,"List(Lower Limb Amputation, Neuropathic Pain)",List(United Kingdom)
NCT04811235,Optical Monitoring With Near-Infrared Spectroscopy for Spinal Cord Injury Trial,"""OMNI-SCI"" Trial - ""Optical Monitoring With Near-Infrared Spectroscopy for Spinal Cord Injury"" Trial",INTERVENTIONAL,List(NA),TERMINATED,1,2022-09-24,2026-04-01,2026-09-23,false,"List(Acute Spinal Cord Injury, Spinal Cord Injuries)",List(Canada)
NCT04208035,Hepatitis C Elimination in the Netherlands,Hepatitis C Elimination in the Netherlands (CELINE) - A National Multicenter Cohort Study Retrieving Lost to Follow-up Chronic Hepatitis C Patients,OBSERVATIONAL,null,UNKNOWN,250,2018-09-17,2021-01-01,2019-12-23,false,List(Chronic Hepatitis c),List(Netherlands)
NCT04382235,Non-invasive Ventilatory Support of Patients Affected by COVID-19,Management Through Non-invasive Ventilatory Support of Patients Showing an Acute Respiratory Failure Related to COVID-19 in Non-intensive Wards,OBSERVATIONAL,null,COMPLETED,909,2020-03-31,2020-05-31,2022-04-28,false,List(Covid-19),List(Italy)


In [0]:
(
    serving_study_search_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "clinical_trials.serving_study_search"
    )
)

In [0]:
# QC
print(
    "Studies:",
    studies_df.count()
)

print(
    "Serving search:",
    serving_study_search_df.count()
)

print(
    "Duplicate NCT:",
    serving_study_search_df
    .groupBy("nct_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Studies: 604566
Serving search: 604566
Duplicate NCT: 0


In [0]:
# Serving study detail
# 특정 study 하나를 Agent나 CRA-RBM에서 열 때 사용할 데이터

# intervention, Oucome, Location을 한 study 단위로
intervention_detail_df = (
    interventions_df
    .groupBy("nct_id")
    .agg(
        F.collect_list(
            F.struct(
                "intervention_order",
                "intervention_type",
                "intervention_name",
                "description",
                "arm_group_labels"
            )
        ).alias("interventions")
    )
)

outcome_detail_df = (
    outcomes_df
    .groupBy("nct_id")
    .agg(
        F.collect_list(
            F.struct(
                "outcome_type",
                "outcome_order",
                "measure",
                "description",
                "time_frame"
            )
        ).alias("outcomes")
    )
)

location_detail_df = (
    locations_df
    .groupBy("nct_id")
    .agg(
        F.collect_list(
            F.struct(
                "location_order",
                "facility",
                "location_status",
                "city",
                "state",
                "country",
                "latitude",
                "longitude"
            )
        ).alias("locations")
    )
)

In [0]:
serving_study_detail_df = (
    studies_df
    .join(
        condition_summary_df,
        on="nct_id",
        how="left"
    )
    .join(
        intervention_detail_df,
        on="nct_id",
        how="left"
    )
    .join(
        outcome_detail_df,
        on="nct_id",
        how="left"
    )
    .join(
        location_detail_df,
        on="nct_id",
        how="left"
    )
)

display(
    serving_study_detail_df.limit(5)
)

nct_id,brief_title,official_title,study_type,phases,overall_status,enrollment_count,enrollment_type,allocation,intervention_model,primary_purpose,masking,start_date,completion_date,last_update_date,has_results,source,ingested_at,conditions,interventions,outcomes,locations
NCT06927635,Stereotactic Body Radiotherapy or Standard of Care for Prostate Oligoprogressive Cancer,Stereotactic Body Radiotherapy or Standard of Care for Prostate Oligoprogressive Cancer: a Randomized Phase II Trial,INTERVENTIONAL,List(PHASE2),RECRUITING,75,ESTIMATED,RANDOMIZED,PARALLEL,TREATMENT,NONE,2025-07-08,2030-06-01,2026-07-07,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z,"List(Castrate Resistance Prostate Cancer, OligoProgressive Metastatic Disease)","List(List(0, RADIATION, Stereotactic Body Radiation Therapy (SBRT), SBRT will be delivered as per institutional standard., List(Arm 1 - Standard of Care, Arm 2 - Experimental)), List(1, OTHER, Systemic therapy, Participants will receive Systemic Therapy. Participants in Arm 1 - SOC may change the Systemic Therapy throughout treatment. Participants in Arm 2 - Experimental will remain on the same Systemic Therapy throughout treatment., List(Arm 1 - Standard of Care, Arm 2 - Experimental)))","List(List(SECONDARY, 0, Radiographic Local Control of Oligopressive Lesions, Radiographic local control of the index OP lesions will be assessed by treating oncologist., Enrollment to 12 months-post enrollment.), List(SECONDARY, 1, Radiographic Distant Control of Oligopressive Lesions, Defined by growth of non-Oligoprogressive metastatic sites and/or the development of new sites of metastatic disease, as assessed by treating oncologist., Enrollment to 12-months post-enrollment.), List(SECONDARY, 2, Progression Free Survival (PFS), Defined as the time from randomization to death from any cause, progression of disease, or date of last follow-up, whichever occurs first. PFS will be assessed by the treating oncologist., Enrollment to 12-months post-enrollment.), List(SECONDARY, 3, Time to next Systemic Therapy, Defined as the time from when the study intervention has been commenced until commencement of any next systemic anti-cancer therapy (including a change to best supportive care) or date of last follow up, whichever occurs first, which will be assess by the treating oncologist., Enrollment to 12-months post-enrollment.), List(SECONDARY, 4, Differences in Toxicity, Toxicity will be assessed to determine whether SBRT compared to SoC results in patient- and/or physician-reported toxicity differences. Toxicity will be collected using CTCAE v5.0, Enrollment to 12-months post-enrollment.), List(SECONDARY, 5, Differences in Quality of Life, Quality of Life (QoL) will be assessed to determine whether SBRT compared to SoC results in patient- and/or physician-reported QoL differences. QoL will be collected using the EORTC QLQ-C30 questionnaire., Enrollment to 12-months post-enrollment.), List(PRIMARY, 0, Study Feasibility, Determine the feasibility of enrolling 21 patients patients onto a study randomizing patients to either MDT using SBRT plus remaining on current ST versus SoC (change in Systemic Therapy) by measuring the rate accrual (number of patients per month for 18 months)., Within 18 months of study activation.), List(PRIMARY, 1, Change in Prostate-Specific Antigen (PSA), Assess by standard PSA blood testing if the proportion of patients having a \>50% change in PSA compared to baseline (PSA50) is improved with MDT using SBRT plus remaining on current ST versus SoC., Enrollment to 12 months post-enrollment.))","List(List(0, Princess Margaret Cancer Center, RECRUITING, Toronto, Ontario, Canada, 43.70643, -79.39864))"
NCT01805635,PF and FeNO and Clinical Characteristics in Children Being Diagnosed Due to Suspicion of Allergic Diseases,Evaluation of the Relationship Between the Parameters of Pulmonary Function (PF) and the Concentration of Fractional Exhaled Nitric Oxide (FeNO) and Clinical Characteristics in Childre

In [0]:
# 특정 NCT ID 조회 테스트

example_nct_id = (
    serving_study_detail_df
    .select("nct_id")
    .first()["nct_id"]
)

print(example_nct_id)

display(
    serving_study_detail_df
    .filter(
        F.col("nct_id") == example_nct_id
    )
)

NCT00634335


nct_id,brief_title,official_title,study_type,phases,overall_status,enrollment_count,enrollment_type,allocation,intervention_model,primary_purpose,masking,start_date,completion_date,last_update_date,has_results,source,ingested_at,conditions,interventions,outcomes,locations
NCT00634335,Echocardiographic Characteristics of High Endurance Israeli Athletes,Echocardiographic Characteristics of High Endurance Israeli Athletes,OBSERVATIONAL,null,UNKNOWN,81,ESTIMATED,null,null,null,null,2008-03-01,2009-06-01,2008-12-30,false,clinicaltrials.gov,2026-09-26T12:15:35.981Z,List(Athlete's Heart),null,null,"List(List(0, Soroka University Medical Center, RECRUITING, Beersheba, null, Israel, 31.25181, 34.7913))"


In [0]:
(
    serving_study_detail_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "clinical_trials.serving_study_detail"
    )
)

print(
    "Serving Detail:",
    serving_study_detail_df.count()
)

print(
    "Duplicate:",
    serving_study_detail_df
    .groupBy("nct_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Serving Detail: 604566
Duplicate: 0
